In [1]:
import numpy as np 
import pandas as pd 
pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.externals import joblib

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)


ValueError: Value must be a nonnegative integer or None

In [2]:
from nltk.tokenize import TweetTokenizer
import datetime
import lightgbm as lgb
from scipy import stats
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split, cross_val_score
from wordcloud import WordCloud
from collections import Counter
from nltk.corpus import stopwords
from nltk.util import ngrams
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier


In [3]:
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer("english")


In [4]:
import spacy


In [5]:
PATH = '../input/'


In [6]:
df_train = pd.read_csv(PATH + "train.tsv", sep = '\t')
df_test = pd.read_csv(PATH + "test.tsv", sep = '\t')


In [7]:
df_train.head(10)


,PhraseId,SentenceId,Phrase,Sentiment
0,24144,1097,through every frame,2
1,85111,4402,onscreen presence,2
2,75999,3900,into dysfunction,2
3,17377,754,a probe,2
4,9664,403,half hour,2
5,120381,6436,in a movie that works against itself,1
6,107838,5703,Canadian filmmaker Gary Burns ' inventive and ...,3
7,151867,8286,from 1998,2
8,94442,4930,"It 's not a bad plot ; but , unfortunately , t...",2
9,32084,1502,Qutting,2


In [8]:
sns.factorplot(x = "Sentiment", data = df_train, kind = 'count', size = 6)


NameError: name 'sns' is not defined

In [9]:
df_train.Sentiment.value_counts()


Sentiment
2    55558
3    23232
1    19134
4     6439
0     4879
Name: count, dtype: int64

In [10]:
print ("Number of sentences is {0:.0f}.".format(df_train.SentenceId.count()))

print ("Number of unique sentences is {0:.0f}.".format(df_train.SentenceId.nunique()))

print ("Number of phrases is {0:.0f}.".format(df_train.PhraseId.count()))


Number of sentences is 109242.
Number of unique sentences is 8503.
Number of phrases is 109242.


In [11]:
print ("The average length of phrases in the training set is {0:.0f}.".format(np.mean(df_train['Phrase'].apply(lambda x: len(x.split(" "))))))

print ("The average length of phrases in the test set is {0:.0f}.".format(np.mean(df_test['Phrase'].apply(lambda x: len(x.split(" "))))))


The average length of phrases in the training set is 7.
The average length of phrases in the test set is 7.


In [12]:
text = ' '.join(df_train.loc[df_train.Sentiment == 0, 'Phrase'].values)


In [13]:
Counter([i for i in ngrams(text.split(), 3)]).most_common(5)


[(('one', 'of', 'the'), 35),
 ((',', 'it', "'s"), 27),
 (('a', 'movie', 'that'), 26),
 (('of', 'a', 'movie'), 25),
 (('of', 'the', 'worst'), 24)]

In [14]:
print (df_train.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109242 entries, 0 to 109241
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   PhraseId    109242 non-null  int64 
 1   SentenceId  109242 non-null  int64 
 2   Phrase      109242 non-null  object
 3   Sentiment   109242 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 3.3+ MB
None


In [15]:
df_train.Phrase.str.len().sort_values(ascending = False)


2880     283
79232    279
86021    266
77505    266
42017    263
        ... 
46442      1
37701      1
20193      1
46703      1
9344       1
Name: Phrase, Length: 109242, dtype: int64

In [16]:
df_train.loc[105155, 'Phrase']


'about other'

In [17]:
df_train.Sentiment.dtype


dtype('int64')

In [18]:
nlp = spacy.load('en',disable=['parser', 'tagger', 'ner'])
def tokenizer(s): 
    return [w.text.lower() for w in nlp(s)]


OSError: [E941] Can't find model 'en'. It looks like you're trying to load a model from a shortcut, which is obsolete as of spaCy v3.0. To load the model, use its full name instead:

nlp = spacy.load("en_core_web_sm")

For more details on the available models, see the models directory: https://spacy.io/models and if you want to create a blank model, use spacy.blank: nlp = spacy.blank("en")

In [19]:
## stemming sentences
sentences = list(df_train.Phrase.values) + list(df_test.Phrase.values)
sentences2 = [[stemmer.stem(word) for word in sentence.split(" ")] for sentence in sentences]
for i in range(len(sentences2)):sentences2[i] = ' '.join(sentences2[i])


In [20]:
tfidf = TfidfVectorizer(strip_accents = 'unicode', tokenizer = tokenizer, encoding='utf-8', ngram_range = (1,2), max_df = 0.75, min_df = 3, sublinear_tf = True)


NameError: name 'tokenizer' is not defined

In [21]:
_ = tfidf.fit(sentences2)


NameError: name 'tfidf' is not defined

In [22]:
train_phrases2 = [[stemmer.stem(word) for word in sentence.split(" ")] for sentence in list(df_train.Phrase.values)]
for i in range(len(train_phrases2)):train_phrases2[i] = ' '.join(train_phrases2[i])
train_df_flags = tfidf.transform(train_phrases2)


NameError: name 'tfidf' is not defined

In [23]:
test_phrases2 = [[stemmer.stem(word) for word in sentence.split(" ")] for sentence in list(df_test.Phrase.values)]
for i in range(len(test_phrases2)):test_phrases2[i] = ' '.join(test_phrases2[i])
test_df_flags = tfidf.transform(test_phrases2)


NameError: name 'tfidf' is not defined

In [24]:
X_train_tf = train_df_flags[0:125000]
X_valid_tf = train_df_flags[125000:]
y_train_tf = (df_train["Sentiment"])[0:125000]
y_valid_tf = (df_train["Sentiment"])[125000:]

print("X_train shape: ", X_train_tf.shape)
print("X_valid shape: ",X_valid_tf.shape)
print("Y_train shape: ",len(y_train_tf))
print("Y_valid shape: ",len(y_valid_tf))


NameError: name 'train_df_flags' is not defined

In [25]:
from sklearn.linear_model import LogisticRegression


In [26]:
scores = cross_val_score(LogisticRegression(C=4, dual=True), X_train_tf, y_train_tf, cv=5)


NameError: name 'X_train_tf' is not defined

In [27]:
scores


NameError: name 'scores' is not defined

In [28]:
np.mean(scores), np.std(scores)


NameError: name 'scores' is not defined

In [29]:
logistic = LogisticRegression(C=4, dual=True)
ovrm = OneVsRestClassifier(logistic)
ovrm.fit(X_train_tf, y_train_tf)


NameError: name 'X_train_tf' is not defined

In [30]:
scores = cross_val_score(ovrm, X_train_tf, y_train_tf, scoring='accuracy', n_jobs=-1, cv=3)


NameError: name 'X_train_tf' is not defined

In [31]:
print (np.mean(scores))
print (np.std(scores))


NameError: name 'scores' is not defined

In [32]:
print ("train accuracy:", ovrm.score(X_train_tf, y_train_tf ))
print ("valid accuracy:", ovrm.score(X_valid_tf, y_valid_tf))


NameError: name 'X_train_tf' is not defined

In [33]:
df_test.head()


,PhraseId,SentenceId,Phrase
0,21924,980,escape movie
1,18207,795,well-executed
2,81803,4221,flick formula
3,114983,6124,is too savvy a filmmaker to let this morph int...
4,135928,7343,"There are laughs aplenty ,"


In [34]:
df_test_logistic = df_test.copy()[["PhraseId"]]
df_test_logistic['Sentiment'] = ovrm.predict(test_df_flags)

df_test_logistic.head()


NameError: name 'test_df_flags' is not defined

In [35]:
svc = LinearSVC(dual=False)
svc.fit(X_train_tf, y_train_tf)


NameError: name 'X_train_tf' is not defined

In [36]:
print ("train accuracy:", svc.score(X_train_tf, y_train_tf ))
print ("valid accuracy:", svc.score(X_valid_tf, y_valid_tf))


NameError: name 'X_train_tf' is not defined

In [37]:
df_test_svm = df_test.copy()[["PhraseId"]]
df_test_svm['Sentiment'] = svc.predict(test_df_flags)
df_test_svm.head()


NameError: name 'test_df_flags' is not defined

In [38]:
df_test_logistic.to_csv("submission_tfidf_logistic.csv", index = False)
